# Módulo 08 — Redes Neurais Sem Peso
**Portfólio de Laboratório de RNA**
**Aluno:** Gabriel Rocha Guimarães | **RA:** 23110134 | **Turma:** 26-1-COMP-7-07-B

---

## Fundamentação Teórica

As **redes neurais sem peso** (weightless neural networks) constituem uma classe fundamentalmente distinta: em vez de multiplicações em ponto flutuante com pesos, operam exclusivamente com **lógica booleana** e **tabelas de lookup em memória RAM**.

**Discriminador de RAM:** cada neurônio é uma memória endereçável. O vetor binário de entrada serve como endereço; a célula apontada armazena 0 ou 1. Treinar significa escrever 1 no endereço correspondente ao padrão apresentado.

**Classificação por votação:** vários discriminadores de RAM trabalham em paralelo. A saída final é o score — a soma dos discriminadores que respondem 1 para o padrão de consulta. Quanto mais discriminadores ativados, maior a confiança de que o padrão pertence à classe treinada.

**Capacidade de memória:** com $n$ bits de entrada, cada discriminador possui $2^n$ células. Isso cresce exponencialmente, limitando $n$ a valores modestos na prática. A vantagem é a velocidade: uma consulta é apenas um acesso à memória, sem nenhuma operação aritmética.

**Treinamento one-shot:** diferente de backpropagation, um único exemplo basta para ser memorizado — não há épocas.

In [1]:
# Passo 2 — Portas lógicas booleanas
def AND(a, b):
    return 1 if (a == 1 and b == 1) else 0

def OR(a, b):
    return 1 if (a == 1 or b == 1) else 0

def NOT(x):
    return 1 - x

print('AND:')
for a in [0, 1]:
    for b in [0, 1]:
        print(f'  {a} AND {b} = {AND(a, b)}')

print('\nOR:')
for a in [0, 1]:
    for b in [0, 1]:
        print(f'  {a} OR {b} = {OR(a, b)}')

print('\nNOT:')
for x in [0, 1]:
    print(f'  NOT {x} = {NOT(x)}')

AND:
  0 AND 0 = 0
  0 AND 1 = 0
  1 AND 0 = 0
  1 AND 1 = 1

OR:
  0 OR 0 = 0
  0 OR 1 = 1
  1 OR 0 = 1
  1 OR 1 = 1

NOT:
  NOT 0 = 1
  NOT 1 = 0


In [2]:
# Passo 3 — Discriminador de RAM
import numpy as np

class RAMDiscriminator:
    def __init__(self, input_size):
        self.input_size = input_size
        self.memory_size = 2 ** input_size
        self.memory = np.zeros(self.memory_size, dtype=int)

    def address_to_index(self, pattern):
        index = 0
        for i, bit in enumerate(pattern):
            index += bit * (2 ** i)
        return int(index)

    def train(self, pattern):
        index = self.address_to_index(pattern)
        self.memory[index] = 1

    def discriminate(self, pattern):
        index = self.address_to_index(pattern)
        return self.memory[index]

ram = RAMDiscriminator(input_size=3)
print('Memória inicial:', ram.memory)

pattern1 = np.array([1, 0, 1])
ram.train(pattern1)
print(f'\nTreinado com {pattern1}')
print('Memória atualizada:', ram.memory)
print(f'\nConsultar {pattern1}: {ram.discriminate(pattern1)}')
print(f'Consultar {np.array([0,1,0])}: {ram.discriminate(np.array([0,1,0]))}')

Memória inicial: [0 0 0 0 0 0 0 0]

Treinado com [1 0 1]
Memória atualizada: [0 0 0 0 0 1 0 0]

Consultar [1 0 1]: 1
Consultar [0 1 0]: 0


In [3]:
# Passo 4 — Rede Sem Peso completa + Classificação de caracteres
import numpy as np

class WeightlessNeuralNetwork:
    def __init__(self, input_size, num_discriminators=None):
        self.input_size = input_size
        if num_discriminators is None:
            num_discriminators = input_size
        self.num_discriminators = num_discriminators
        self.rams = [RAMDiscriminator(input_size) for _ in range(num_discriminators)]

    def train(self, patterns):
        for pattern in patterns:
            for ram in self.rams:
                ram.train(pattern)

    def predict(self, pattern):
        score = 0
        for ram in self.rams:
            score += ram.discriminate(pattern)
        return score

    def classify(self, pattern, threshold=0.5):
        score = self.predict(pattern)
        return 1 if score >= (self.num_discriminators * threshold) else 0

# Caracteres 3x3 (9 bits)
A = np.array([0, 1, 0, 1, 0, 1, 1, 1, 1])
B = np.array([1, 1, 0, 1, 1, 0, 1, 1, 1])
C = np.array([0, 1, 1, 1, 0, 0, 1, 1, 1])

wnn_A = WeightlessNeuralNetwork(input_size=9); wnn_A.train([A])
wnn_B = WeightlessNeuralNetwork(input_size=9); wnn_B.train([B])
wnn_C = WeightlessNeuralNetwork(input_size=9); wnn_C.train([C])

print('CLASSIFICAÇÃO DE CARACTERES (3x3 pixels):')
for char, label in [(A,'A'),(B,'B'),(C,'C')]:
    scores = {'A': wnn_A.predict(char), 'B': wnn_B.predict(char), 'C': wnn_C.predict(char)}
    predicted = max(scores, key=scores.get)
    print(f'  Char {label}: scores={scores} → Predito: {predicted} | OK: {predicted==label}')

print('\nTeste com exemplo geral (padrões positivos):')
positive_patterns = [np.array([1,1,1,1]), np.array([1,0,1,1]), np.array([1,1,0,1])]
wnn = WeightlessNeuralNetwork(input_size=4)
wnn.train(positive_patterns)
for pat in [np.array([1,1,1,1]), np.array([0,0,0,0])]:
    score = wnn.predict(pat)
    print(f'  {pat}: score={score}/{wnn.num_discriminators} → {"POSITIVO" if wnn.classify(pat) else "NEGATIVO"}')

CLASSIFICAÇÃO DE CARACTERES (3x3 pixels):
  Char A: scores={'A': np.int64(9), 'B': np.int64(0), 'C': np.int64(0)} → Predito: A | OK: True
  Char B: scores={'A': np.int64(0), 'B': np.int64(9), 'C': np.int64(0)} → Predito: B | OK: True
  Char C: scores={'A': np.int64(0), 'B': np.int64(0), 'C': np.int64(9)} → Predito: C | OK: True

Teste com exemplo geral (padrões positivos):
  [1 1 1 1]: score=4/4 → POSITIVO
  [0 0 0 0]: score=0/4 → NEGATIVO


## Análise Crítica

**One-shot learning:** O ponto mais distintivo das redes sem peso é que uma única apresentação de cada padrão é suficiente. Não existe convergência iterativa — o padrão é gravado diretamente na RAM em O(1).

**Classificação exata vs. generalização:** Para padrões idênticos aos treinados, o score é máximo. Para padrões diferentes, o score é 0 — a rede não generaliza para variações. Isso é exatamente o oposto do que o MLP faz via gradiente.

**Crescimento exponencial de memória:** Com $n$ bits de entrada, cada discriminador precisa de $2^n$ células. Para entradas de imagem (mesmo pequenas, como 28x28 = 784 bits) isso seria computacionalmente inviável. A solução prática é particionar a entrada em blocos menores.

**Aplicações ideais:** Sistemas onde velocidade é crítica e os padrões são conhecidos com antecedência — controle em tempo real, reconhecimento de padrões binários fixos, sistemas embarcados sem unidade de ponto flutuante.